# 🔍 Opening the Black Box of an LLM

### A hands-on tour of GPT-2, the grandparent of ChatGPT

People call large language models (LLMs) "magic" or a "black box."

In the next few minutes you'll open that box yourself. You'll download the real
GPT-2 model — the same kind of model behind ChatGPT, just older and smaller —
and watch, step by step, exactly what happens when it reads text and writes the
next word.

**No math background needed.** If you can press a button, you can do this.

---

### How to use this notebook

- Each grey box below is a **cell**. Click a cell, then press **`Shift + Enter`** to run it.
- Run the cells **in order, from top to bottom.**
- You don't need to understand the code — just run it and read what comes out.
- The whole thing runs for free in your browser. Nothing installs on your computer.

▶️ **Start by running the cell right below this one.**


## Step 0 — Get the tools ready

This cell loads two free, open-source tools:

- **`transformers`** — a library from Hugging Face that lets us download and run the model.
- **`torch`** (PyTorch) — the engine that does the number-crunching.

In [1]:
import torch
import torch.nn.functional as F

print("✅ Ready to go!")
print("PyTorch version:", torch.__version__)

✅ Ready to go!
PyTorch version: 2.13.0


## Step 1 — Download the "brain": it's just a big pile of numbers

An LLM's entire knowledge — grammar, facts, style, everything it "learned" — is
stored as a giant list of numbers called **parameters** (also called **weights**).

Let's download GPT-2 and see them.


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Download GPT-2 (the small version). This grabs the model + its tokenizer.
# First run takes ~10-20 seconds because it downloads the file.
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

model.eval()  # put the model in "just answer questions" mode
print("✅ GPT-2 downloaded.")

/Users/aucheng/code/play/colab-gpt2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 54610.45it/s]


✅ GPT-2 downloaded.


### So how many numbers is "a lot"?

Let's count every parameter in the model.


In [3]:
total_parameters = sum(p.numel() for p in model.parameters())

print(f"GPT-2 contains {total_parameters:,} numbers.")
print(f"That's about {total_parameters/1_000_000:.0f} million parameters.")

GPT-2 contains 124,439,808 numbers.
That's about 124 million parameters.


> 💡 **Concept — Parameters (weights)**
>
> Those ~124 million numbers *are* the model. There is no hidden rulebook, no
> database of answers, no human in the loop. Just numbers, organized into grids
> called **weight matrices**.
>
> (For scale: the model behind ChatGPT has **hundreds of billions** of these
> numbers — over a thousand times more.)

Let's actually *look* at one of those weight matrices. This is a real slice of
GPT-2's brain:


In [4]:
# Pull out one weight matrix from deep inside the model and print it.
one_weight_matrix = model.transformer.h[0].attn.c_attn.weight

print("Shape (rows x columns):", tuple(one_weight_matrix.shape))
print()
print(one_weight_matrix)

Shape (rows x columns): (768, 2304)

Parameter containing:
tensor([[-0.4738, -0.2614, -0.0978,  ...,  0.0513, -0.0584,  0.0250],
        [ 0.0874,  0.1473,  0.2387,  ..., -0.0525, -0.0113, -0.0156],
        [ 0.0039,  0.0695,  0.3668,  ...,  0.1143,  0.0363, -0.0318],
        ...,
        [-0.2592, -0.0164,  0.1991,  ...,  0.0095, -0.0516,  0.0319],
        [ 0.1517,  0.2170,  0.1043,  ...,  0.0293, -0.0429, -0.0475],
        [-0.4100, -0.1924, -0.2400,  ..., -0.0046,  0.0070,  0.0198]],
       requires_grad=True)


That's it. **That wall of decimals is what "learning" looks like inside a
machine.** The model has thousands of grids like this. When people say a model
was "trained," they mean: a computer slowly tuned every one of these numbers
until the model got good at predicting text.

Our job now is to watch these numbers do something useful.


## Step 2 — Turn a sentence into tokens

A model can't read letters — it only does math on numbers. So the first thing
that happens to your text is it gets chopped into pieces called **tokens**, and
each token gets an ID number.

Let's tokenize a sentence and see the pieces.


In [5]:
sentence = "Machine learning is surprisingly simple."

# Break the sentence into tokens (word-pieces).
token_ids = tokenizer.encode(sentence)
token_strings = [tokenizer.decode([t]) for t in token_ids]

print("Original sentence:")
print("  ", sentence)
print()
print("Broken into", len(token_ids), "tokens:")
for token_text, token_id in zip(token_strings, token_ids):
    print(f"   '{token_text}'   ->   ID {token_id}")

Original sentence:
   Machine learning is surprisingly simple.

Broken into 6 tokens:
   'Machine'   ->   ID 37573
   ' learning'   ->   ID 4673
   ' is'   ->   ID 318
   ' surprisingly'   ->   ID 12362
   ' simple'   ->   ID 2829
   '.'   ->   ID 13


> 💡 **Concept — What is a token?**
>
> A **token** is a chunk of text — sometimes a whole word, sometimes just a
> piece of one (notice how a longer word can split into parts). GPT-2 knows about
> 50,000 different tokens, each with its own ID number.
>
> Everything the model does, it does with these ID numbers. **"Tokens" are the
> only language the model actually speaks.**

> 💡 **Concept — Context size**
>
> A model can only look at so many tokens at once — its **context size**. GPT-2's
> limit is **1024 tokens** (roughly 750 words). Modern models can handle hundreds
> of thousands. Anything past the limit, the model simply can't see.


In [6]:
print("GPT-2's vocabulary:", f"{tokenizer.vocab_size:,} different tokens")
print("GPT-2's context size:", model.config.n_positions, "tokens at a time")

GPT-2's vocabulary: 50,257 different tokens
GPT-2's context size: 1024 tokens at a time


## Step 3 — Turn each token into a vector of numbers

An ID number like `4572` is just a name tag — it doesn't tell the model anything
about *meaning*. So each token ID is swapped for a list of numbers called an
**embedding vector**. Think of it as the token's "coordinates" in the model's
mind.

Let's look at the embedding vector for a single word.


In [7]:
word = " king"   # (the space matters to GPT-2 — it's part of the token)

token_id = tokenizer.encode(word)[0]
embedding_vector = model.transformer.wte.weight[token_id]

print(f"The word '{word.strip()}' becomes this list of {embedding_vector.shape[0]} numbers:")
print()
print(embedding_vector.detach().numpy())

The word 'king' becomes this list of 768 numbers:

[-0.01946725 -0.11748341  0.08894998  0.07266869  0.16484568 -0.01115442
 -0.32727474  0.05089554  0.032837   -0.10069963  0.17811741 -0.05375019
  0.00346752  0.02685049  0.2226163  -0.08362372  0.07665508  0.08152209
  0.01222507  0.13032244  0.1057746   0.04661812  0.03033879  0.13394257
  0.13063034  0.13872057  0.07950757  0.00788677  0.10417033 -0.10904862
 -0.03000498 -0.11496009  0.06010049  0.11254642 -0.06759521  0.10113526
 -0.3154677  -0.04903419  0.11070949 -0.04783219 -0.01520181  0.14083312
  0.00514873 -0.07778836 -0.14498846 -0.00955169  0.25265703 -0.32691514
  0.15923409 -0.10724931 -0.07622112 -0.07068396  0.19758096 -0.06014417
 -0.10696468 -0.14764206  0.09665486 -0.1190175   0.01141881 -0.0343446
 -0.16817175  0.01271043 -0.14529695  0.292693    0.11044775  0.15025426
 -0.09988158 -0.02045775  0.10850653 -0.03897435  0.27047348 -0.08171062
  0.04198844 -0.03428513  0.05121883  0.13401122  0.15864314  0.13616702
 

Every token turns into a vector like this — GPT-2 uses **768 numbers** per token.
So your sentence from Step 2 becomes a **stack of vectors**, one row per token:


In [8]:
sentence = "Machine learning is surprisingly simple."
input_ids = tokenizer.encode(sentence, return_tensors="pt")

# Look up the embedding vector for every token at once.
stack_of_vectors = model.transformer.wte(input_ids)[0]

print("Your sentence is now a grid of numbers:")
print("   rows  =", stack_of_vectors.shape[0], "(one per token)")
print("   columns =", stack_of_vectors.shape[1], "(numbers per token)")
print()
print(stack_of_vectors.detach().numpy())

Your sentence is now a grid of numbers:
   rows  = 6 (one per token)
   columns = 768 (numbers per token)

[[-0.01636353 -0.11124826  0.23272906 ...  0.12077022  0.01844326
  -0.18808651]
 [ 0.02383363  0.00705459  0.05322737 ... -0.10096545  0.01217919
  -0.00309868]
 [-0.00972351  0.01005133  0.05557884 ...  0.11449675 -0.03800575
  -0.02537755]
 [-0.03984132  0.02428987  0.07176195 ...  0.03690743 -0.14133236
  -0.01103566]
 [ 0.14170375  0.07546638  0.09681763 ...  0.23554868  0.00462168
  -0.18926038]
 [ 0.04664114 -0.01130235  0.02832778 ... -0.07351072  0.04961816
   0.09631975]]


> 💡 **Concept — Everything is vectors**
>
> Your sentence is now a grid of numbers — nothing more. From here on, the model
> only ever does one kind of thing: **multiply and add these numbers together,
> over and over.** That's the "thinking."


## Step 4 — Do a *huge* amount of arithmetic

Now the model takes that grid of vectors and pushes it through **12 layers** of
computation. In each layer it does two things:

1. **Mixes the vectors together** so each word can "look at" the others and pick
   up context (this is the famous *attention* mechanism).
2. **Runs every vector through those weight matrices** we saw in Step 1.

This is millions of multiply-and-add operations. It's the "large amount of
computation" that makes the whole thing feel like magic — but it's just
arithmetic with the numbers from Step 1.

The final result is a single **logit vector**: one score for every possible next
token in the vocabulary.


In [9]:
sentence = "The capital of France is"
input_ids = tokenizer.encode(sentence, return_tensors="pt")

with torch.no_grad():          # we're just running it, not training
    output = model(input_ids)

logits = output.logits[0, -1, :]   # scores for the token that comes NEXT

print("The model looked at:", repr(sentence))
print()
print("It produced one score (a 'logit') for EVERY possible next token.")
print("That's a vector of", logits.shape[0], "numbers:")
print()
print(logits.detach().numpy())

The model looked at: 'The capital of France is'

It produced one score (a 'logit') for EVERY possible next token.
That's a vector of 50257 numbers:

[-108.95419 -108.9326  -112.57931 ... -118.33448 -113.15045 -110.37786]


> 💡 **Concept — What is "inference"?**
>
> Running text through the model to get an answer is called **inference**. That's
> all a chatbot does when it replies to you: inference, one step at a time. (The
> other phase — *training* — is the slow, expensive process that set those
> parameters in the first place. We're not doing that here.)

That logit vector is the model's raw opinion. But raw scores are hard to read —
let's turn them into something human.


## Step 5 — From scores to a prediction (and it's a *guess*)

We convert the logit scores into **probabilities** — percentages that add up to
100%. Then we can see the model's top guesses for the next word.


In [10]:
probabilities = F.softmax(logits, dim=-1)   # turn scores into percentages

top10 = torch.topk(probabilities, 10)

print(f"After '{sentence}', GPT-2's top 10 guesses for the next token:")
print()
for prob, token_id in zip(top10.values, top10.indices):
    token_text = tokenizer.decode([token_id])
    bar = "#" * int(prob.item() * 100)
    print(f"   {prob.item()*100:5.1f}%  '{token_text}'  {bar}")

After 'The capital of France is', GPT-2's top 10 guesses for the next token:

     8.5%  ' the'  ########
     4.8%  ' now'  ####
     4.6%  ' a'  ####
     3.2%  ' France'  ###
     3.2%  ' Paris'  ###
     2.7%  ' in'  ##
     2.6%  ' also'  ##
     2.4%  ' not'  ##
     2.3%  ' home'  ##
     1.6%  ' still'  #


> 💡 **Concept — Next-token prediction is probabilistic**
>
> The model doesn't "know" the answer. It produces a **probability for every
> possible next token** and then picks one — often the top guess, but sometimes
> it rolls the dice among the likely options.
>
> This is why an LLM can give you a different answer each time, and why it's
> really doing one simple thing on repeat: **guess the next token.**

**A full sentence is just this one step, done again and again:** guess a token,
add it to the text, then guess the next one. Let's watch that happen live.


## Step 6 — The inference loop: writing one token at a time

Here is the entire "AI writing" trick, in a simple `for` loop:

1. Feed the current text to the model.
2. Get the probabilities for the next token.
3. Pick one token.
4. Stick it on the end of the text.
5. Repeat.

Run the cell and watch GPT-2 write, token by token, in real time. 👇


In [11]:
import sys

prompt = "The secret to a happy life is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

print(prompt, end="")
sys.stdout.flush()

for step in range(40):                       # generate 40 tokens
    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]

    # Turn scores into probabilities (temperature 0.8 = a little creativity).
    probabilities = F.softmax(logits / 0.8, dim=-1)

    # Roll the dice among the likely tokens to pick the next one.
    next_token = torch.multinomial(probabilities, num_samples=1)

    # Add the new token to the text and show it.
    input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
    print(tokenizer.decode(next_token), end="")
    sys.stdout.flush()

print("\n\n✅ Done. GPT-2 just wrote that, one token at a time.")

The secret to a happy life is a quality that keeps people going.

That type of optimism can be found in many Muslim countries, including India, where poverty and lack of stability make life difficult in many communities.

For

✅ Done. GPT-2 just wrote that, one token at a time.


Try running that cell a few times — you'll get a **different result each time**,
because of the dice-roll in step 3. That randomness is exactly the "probabilistic"
idea from Step 5.

Want to put words in GPT-2's mouth? Change `prompt` above to anything you like
and run it again. (GPT-2 is from 2019 and fairly small, so expect it to be a bit
goofy — that's part of the fun.)


## Step 7 — Why is this slow? Meet the GPU

You might have noticed the loop above was a little sluggish. That's because it's
running on a **CPU** — the general-purpose chip in every computer, which does
math mostly one step at a time.

Let's time how long the model takes right now.


In [12]:
import time

where = "GPU 🚀" if torch.cuda.is_available() else "CPU 🐢"
print("Currently running on:", where)
print()

prompt = "In the year 2050, computers will"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
if torch.cuda.is_available():
    model.to("cuda")
    input_ids = input_ids.to("cuda")

start = time.time()
with torch.no_grad():
    for _ in range(50):
        logits = model(input_ids).logits[0, -1, :]
        next_token = torch.multinomial(F.softmax(logits, dim=-1), 1)
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
elapsed = time.time() - start

print(f"Generated 50 tokens in {elapsed:.2f} seconds "
      f"({50/elapsed:.1f} tokens per second).")

Currently running on: CPU 🐢

Generated 50 tokens in 1.24 seconds (40.3 tokens per second).


> 💡 **Concept — Why GPUs?**
>
> Remember Step 4 was *millions* of multiply-and-add operations. A **GPU**
> (graphics processing unit) is a chip built to do thousands of those
> multiplications **at the same time** instead of one after another.
>
> Because an LLM is basically one giant pile of multiplications, a GPU can run it
> **many times faster** than a CPU. This is the entire reason companies buy
> warehouses full of GPUs to run AI.

### 🚀 Try it yourself: switch on the GPU

Colab gives you a free GPU. Turn it on and re-run to feel the difference:

1. In the top menu, click **Runtime → Change runtime type**.
2. Under **Hardware accelerator**, choose **T4 GPU**, then click **Save**.
   (Colab will restart — that's normal.)
3. Now re-run the notebook from the top: **Runtime → Run all**.

When you reach this cell again, it should say **"running on: GPU 🚀"** and
generate tokens noticeably faster. Same model, same numbers — just a chip built
for the job.


## 🎉 You opened the black box

Here's the whole thing you just saw, start to finish:

| Step | What happened | The big idea |
|------|---------------|--------------|
| 1 | Downloaded ~124 million numbers | The model **is** its parameters (weights) |
| 2 | Sentence → tokens | Models read **tokens**, not letters |
| 3 | Tokens → vectors | Every token becomes a list of numbers |
| 4 | A huge amount of arithmetic | This is **inference** — just multiply & add |
| 5 | Scores → probabilities | Prediction is a **probabilistic guess** |
| 6 | A `for` loop | Text is built **one token at a time** |
| 7 | CPU vs GPU | **GPUs** do the math massively in parallel |

There was no magic anywhere — just a very large amount of very simple arithmetic
on a very large pile of numbers.

The models powering today's chatbots work **exactly** like this. They're bigger
(billions of parameters, longer context), trained on far more text, and polished
with extra steps — but the core loop you just watched is the same one.

**You've now seen inside the black box.** 🔓

---

*Curious to go further? Try changing the prompts, generate longer text, or load a
bigger model by swapping `"gpt2"` for `"gpt2-medium"` or `"gpt2-large"` in Step 1.*
